# Running the Patch-Clamp Pipeline Locally

### **Overview**

This notebook runs the full patch-clamp electrophysiology pipeline for registered experiments. It processes ABF recordings through feature extraction, plot generation, and report table population.

**Prerequisites:**
- Experiment registered in `EphysExperimentsForAnalysis` (see [CREATE_patch_clamp_experiment.ipynb](./CREATE_patch_clamp_experiment.ipynb))
- ABF files and Excel metadata available locally (downloaded from S3 or local)
- `ffmpeg` installed for animated GIF/MP4 generation (`brew install ffmpeg` on macOS)

### **Pipeline Stages**

| Stage | Table | Description |
|-------|-------|-------------|
| 1 | `Animals` | Parse animal/strain metadata from Excel |
| 2 | `PatchCells` | Register patched cells per session |
| 3 | `EphysRecordings` | Register individual ABF recordings per cell |
| 4 | `APandIntrinsicProperties` | Extract AP features (threshold, firing rate, input resistance, etc.) |
| 5 | `CurrentStepPlots` | Generate traces, F-I, V-I, spike, phase, derivative plots |
| 6 | Individual plot tables | `FICurvePlots`, `VICurvePlots`, `FirstSpikePlots`, etc. |
| 7 | `CombinedPlotsWithText` | Multi-panel summary plot |
| 8 | `AnimatedCurrentStepPlots` | Animated GIF + MP4 per recording |
| 9 | `PatchClampReport` | Convert filepath plots to `attach` format for dashboard |

#### **Setup**

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
import datajoint as dj
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for plot generation

In [ ]:
from workflow.pipeline.patch_clamp_ephys import schema_ephys as patch_clamp
from workflow.pipeline import report

#### **Check Registered Experiments**

Verify which experiments are registered and ready for processing.

In [ ]:
patch_clamp.EphysExperimentsForAnalysis()

In [ ]:
patch_clamp.CurrentStepTimeParams()

#### **Step 1: Populate Metadata Tables**

These tables parse the Excel metadata file for each experiment to extract animal info, cell records, and recording entries.

In [ ]:
patch_clamp.Animals.populate(display_progress=True)
patch_clamp.PatchCells.populate(display_progress=True)
patch_clamp.EphysRecordings.populate(display_progress=True)

In [ ]:
# Check how many recordings were registered
print(f"Experiments: {len(patch_clamp.EphysExperimentsForAnalysis())}")
print(f"Animals: {len(patch_clamp.Animals())}")
print(f"Cells: {len(patch_clamp.PatchCells())}")
print(f"Recordings: {len(patch_clamp.EphysRecordings())}")

#### **Step 2: Extract Electrophysiology Features**

`APandIntrinsicProperties` reads each ABF file, runs current-step analysis, and extracts features such as AP threshold, input resistance, max firing rate, and F-I curve slope.

In [ ]:
patch_clamp.APandIntrinsicProperties.populate(display_progress=True)

#### **Step 3: Generate Plots**

This populates all plot tables: current step traces, F-I curves, V-I curves, spike waveforms, phase planes, derivative plots, combined summary plots, and animated GIFs.

**Note:** This step generates PNG/GIF files on disk. `CurrentStepPlots` alone creates 6 files per recording.

In [ ]:
# Core plots (traces + 5 feature plots per recording)
patch_clamp.CurrentStepPlots.populate(display_progress=True)

# Individual feature plot tables
patch_clamp.FICurvePlots.populate(display_progress=True)
patch_clamp.VICurvePlots.populate(display_progress=True)
patch_clamp.FirstSpikePlots.populate(display_progress=True)
patch_clamp.PhasePlanes.populate(display_progress=True)
patch_clamp.FirstSpikeFirstDerivativePlots.populate(display_progress=True)
patch_clamp.FirstSpikeSecondDerivativePlots.populate(display_progress=True)

# Combined multi-panel summary plots
patch_clamp.CombinedPlots.populate(display_progress=True)
patch_clamp.CombinedPlotsWithText.populate(display_progress=True)

# Animated current step traces (requires ffmpeg for MP4)
patch_clamp.AnimatedCurrentStepPlots.populate(display_progress=True)

#### **Step 4: Populate Report Tables for Dashboard**

`PatchClampReport` reads the file paths from the plot tables above and stores the actual image files as binary attachments (`attach` type) in the database. This is required for the dashboard `PlotGrid` component to render images.

In [ ]:
report.PatchClampReport.populate(display_progress=True)

#### **Step 5: Verify Results**

In [ ]:
print("=== Pipeline Summary ===")
print(f"Experiments:               {len(patch_clamp.EphysExperimentsForAnalysis())}")
print(f"Recordings:                {len(patch_clamp.EphysRecordings())}")
print(f"AP & Intrinsic Properties: {len(patch_clamp.APandIntrinsicProperties())}")
print(f"  - With APs:             {len(patch_clamp.APandIntrinsicProperties & 'has_ap = \"Yes\"')}")
print(f"  - Without APs:          {len(patch_clamp.APandIntrinsicProperties & 'has_ap = \"No\"')}")
print(f"CurrentStepPlots:          {len(patch_clamp.CurrentStepPlots())}")
print(f"AnimatedCurrentStepPlots:  {len(patch_clamp.AnimatedCurrentStepPlots())}")
print(f"CombinedPlotsWithText:     {len(patch_clamp.CombinedPlotsWithText())}")
print()
print("=== Report Tables (Dashboard) ===")
print(f"PatchClampReport (master):     {len(report.PatchClampReport())}")
print(f"  FICurve:                     {len(report.PatchClampReport.FICurve())}")
print(f"  VICurve:                     {len(report.PatchClampReport.VICurve())}")
print(f"  FirstSpike:                  {len(report.PatchClampReport.FirstSpike())}")
print(f"  PhasePlane:                  {len(report.PatchClampReport.PhasePlane())}")
print(f"  CurrentStep:                 {len(report.PatchClampReport.CurrentStep())}")
print(f"  FirstSpikeDerivative:        {len(report.PatchClampReport.FirstSpikeDerivative())}")
print(f"  FirstSpikeSecondDerivative:  {len(report.PatchClampReport.FirstSpikeSecondDerivative())}")
print(f"  CombinedPlot:                {len(report.PatchClampReport.CombinedPlot())}")
print(f"  AnimatedTrace:               {len(report.PatchClampReport.AnimatedTrace())}")

### **Next Steps**

- View results in the dashboard under **Images & Plots > Patch-Clamp** pages
- Explore results programmatically using [EXPLORE_patch_clamp.ipynb](./EXPLORE_patch_clamp.ipynb)
- Re-upload data to S3 if running locally (use Axon)